<a href="https://colab.research.google.com/github/barryvv/OTN-simulator/blob/main/train_grpo_scratch_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# From-Scratch GRPO on OTN — Colab Runner

End-to-end Colab notebook for training the from-scratch GRPO trainer
(`grpo_scratch/`) on a Colab CUDA GPU. Use this when MPS on a Mac is
too slow for autoregressive sampling.

**Recommended Colab runtime:** T4 (free) or L4/A100 (Colab Pro)
- T4 (16 GB): fits Qwen 2.5-1.5B comfortably; Qwen 2.5-3B with bf16 + LoRA + grad checkpointing
- L4 / A100 (24+ GB): fits Qwen 2.5-3B at full speed; Qwen 2.5-7B with grad checkpointing

Set runtime: **Runtime → Change runtime type → GPU**.

## 1. Verify GPU + install deps

In [1]:
!nvidia-smi

Thu Jun  4 01:11:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   29C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
%pip install -q "torch>=2.2" "transformers>=4.40" "peft>=0.10" "datasets>=2.14" "accelerate>=0.30" pytest
# Colab ships torchao 0.10 which is below the version PEFT's LoRA dispatcher
# requires (>=0.16). We do not use torchao, so uninstalling makes PEFT skip
# that branch cleanly.
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


## 2. Clone the repo

In [3]:
import os, pathlib
REPO_DIR = '/content/OTN-simulator'
if not pathlib.Path(REPO_DIR).exists():
    !git clone https://github.com/barryvv/OTN-simulator.git {REPO_DIR}
%cd {REPO_DIR}
!ls grpo_scratch/

Cloning into '/content/OTN-simulator'...
remote: Enumerating objects: 162, done.
remote: Counting objects: 100% (162/162), done.
remote: Compressing objects: 100% (134/134), done.
remote: Total 162 (delta 38), reused 149 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (162/162), 343.12 KiB | 10.09 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/OTN-simulator
advantages.py  log_probs.py  rewards_continuous.py  sampling.py
__init__.py    losses.py     rewards.py		    trainer.py


## 3. Upload `train.jsonl`

The training dataset (`outputs/grpo_data/train.jsonl`, ~2.8 MB) is gitignored
in this public repo. Run the cell below and select your local copy.

*Alternative:* mount Google Drive and copy from there. See the optional Drive cell.

In [4]:
from google.colab import files
uploaded = files.upload()           # pick train.jsonl from local disk
import os, shutil, pathlib
for name in uploaded:
    target = pathlib.Path('outputs/grpo_data/train.jsonl')
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(name, target)
    print(f'[ok] {name} -> {target} ({target.stat().st_size/1e6:.2f} MB)')

Saving train.jsonl to train.jsonl
[ok] train.jsonl -> outputs/grpo_data/train.jsonl (2.94 MB)


In [5]:
# Optional: load from Drive instead of uploading. Uncomment and adjust the path.
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil, pathlib
# src = pathlib.Path('/content/drive/MyDrive/path/to/train.jsonl')
# dst = pathlib.Path('outputs/grpo_data/train.jsonl')
# dst.parent.mkdir(parents=True, exist_ok=True)
# shutil.copy(src, dst)
# print('[ok] copied:', dst, dst.stat().st_size)

## 4. Run the unit tests (proof the math is right)

Six tests covering the group-relative advantage standardization,
clipped surrogate loss, KL toggle, and log-prob shift+mask.

In [6]:
!python -m pytest tests/test_grpo_scratch.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/OTN-simulator
plugins: langsmith-0.8.5, anyio-4.13.0, typeguard-4.5.2
collected 6 items                                                              

tests/test_grpo_scratch.py::test_group_relative_advantages_mean_zero_std_one PASSED [ 16%]
tests/test_grpo_scratch.py::test_group_relative_advantages_constant_group_does_not_blow_up PASSED [ 33%]
tests/test_grpo_scratch.py::test_group_relative_advantages_rejects_bad_batch PASSED [ 50%]
tests/test_grpo_scratch.py::test_grpo_loss_clipping_activates_for_large_ratios PASSED [ 66%]
tests/test_grpo_scratch.py::test_grpo_loss_kl_only_applied_when_beta_positive PASSED [ 83%]
tests/test_grpo_scratch.py::test_compute_log_probs_shift_and_mask PASSED [100%]

============================== 6 passed in 3.66s ===============================


## 5. Train

Defaults below target a T4 (16 GB) with Qwen 2.5-3B + LoRA + bf16 +
gradient checkpointing. Tune `--group-size`, `--max-new-tokens`, and
`--max-steps` based on your runtime.

In [7]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u train_grpo_scratch.py \
  --base-model Qwen/Qwen2.5-3B-Instruct \
  --train-file outputs/grpo_data/train.jsonl \
  --output-dir /content/adapters/qwen-3b-grpo-scratch \
  --device cuda \
  --dtype bfloat16 \
  --group-size 4 \
  --max-new-tokens 1024 \
  --max-steps 50 \
  --lr 5e-5 \
  --temperature 1.3 \
  --max-grad-norm 10.0 \
  --reward-fn continuous \
  --gradient-checkpointing \
  --empty-cache-between-phases

[info] device=cuda  dtype=torch.bfloat16
[info] base model: Qwen/Qwen2.5-3B-Instruct
[info] train file: outputs/grpo_data/train.jsonl
config.json: 100% 661/661 [00:00<00:00, 5.59MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 40.3MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 109MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 166MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 255MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 35.6k/35.6k [00:00<00:00, 173MB/s]
Fetching 2 files: 100% 2/2 [00:09<00:00,  4.79s/it]
Download complete: 100% 6.17G/6.17G [00:09<00:00, 633MB/s]
Loading weights: 100% 434/434 [00:00<00:00, 1029.45it/s, Materializing param=model.norm.weight]
generation_config.json: 100% 242/242 [00:00<00:00, 3.25MB/s]
trainable params: 7,372,800 || all params: 3,093,311,488 || trainable%: 0.2383
[info] using continuous (similarity-based) reward functions
Generating train split: 147 examples [00:00, 2958.10 examples/s]


### Optional: bigger model (L4 / A100 only)

If you have a Colab Pro L4 (24 GB) or A100 (40 GB), swap to Qwen 2.5-7B
for the production target. Keep `--gradient-checkpointing` on the L4.

In [8]:
# !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u train_grpo_scratch.py \
#   --base-model Qwen/Qwen2.5-7B-Instruct \
#   --train-file outputs/grpo_data/train.jsonl \
#   --output-dir /content/adapters/qwen-7b-grpo-scratch \
#   --device cuda --dtype bfloat16 \
#   --group-size 4 --max-new-tokens 1024 --max-steps 50 --lr 5e-6 --temperature 1.0 \
#   --gradient-checkpointing --empty-cache-between-phases

## 6. Download or persist the adapter

Two paths: zip and download to your laptop, or copy to Drive.

In [9]:
import shutil, pathlib
adapter_dir = pathlib.Path('/content/adapters/qwen-3b-grpo-scratch/final')
if adapter_dir.exists():
    zip_path = shutil.make_archive('/content/grpo-scratch-final', 'zip', adapter_dir)
    from google.colab import files
    files.download(zip_path)
else:
    print(f'[warn] {adapter_dir} not found; training may still be running.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
# Optional: copy adapter to Drive.
# from google.colab import drive; drive.mount('/content/drive')
# import shutil, pathlib
# src = pathlib.Path('/content/adapters/qwen-3b-grpo-scratch/final')
# dst = pathlib.Path('/content/drive/MyDrive/grpo-scratch-final')
# shutil.copytree(src, dst, dirs_exist_ok=True)
# print('[ok] saved to', dst)